# PyBiRewireX — Tutorial

**PyBiRewireX** is a Python port of the [BiRewire R package](https://bioconductor.org/packages/BiRewire/).

Given a binary network (bipartite or undirected), it generates randomised versions
drawn uniformly from the set of all networks with the **same degree sequence**.
This is the standard null model for testing whether observed network properties
are driven by degree alone.

**Algorithm:** Switching Algorithm (SA) — at each step, two edges are chosen at
random and their endpoints swapped if the result is a valid new edge.
After N switching steps the distribution is provably close to uniform.

---

## Contents

1. [Setup](#1-setup)
2. [Create networks](#2-create-networks)
3. [Bipartite rewiring](#3-bipartite-rewiring)
4. [Analytical bound N](#4-analytical-bound-n)
5. [Convergence analysis — bipartite](#5-convergence-analysis--bipartite)
6. [Undirected rewiring](#6-undirected-rewiring)
7. [Convergence analysis — undirected](#7-convergence-analysis--undirected)
8. [Input format support](#8-input-format-support)
9. [Null-model sampler](#9-null-model-sampler)
10. [Markov chain monitoring](#10-markov-chain-monitoring)

## 1. Setup

```bash
pip install pybirewirex            # core (numpy + scipy + C extension)
pip install "pybirewirex[graph]"   # + igraph / networkx support
pip install "pybirewirex[vis]"     # + matplotlib / scikit-learn for plots
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from scipy.stats import t as t_dist

import pybirewirex as pbr
from pybirewirex.similarity import jaccard       # Jaccard index between two networks
from pybirewirex._bounds import bound_bipartite  # analytical bound formula

# _C_AVAILABLE is True when the compiled C extension is loaded.
# If False, a pure-NumPy fallback is used — correct but slower.
print(f"C backend: {pbr._C_AVAILABLE}")

## 2. Create networks

We use the same synthetic networks as the BiRewire R vignette:

```r
ggg   <- bipartite.random.game(n1=100, n2=40, p=0.2)
g.und <- erdos.renyi.game(directed=F, loops=F, n=200, p.or.m=0.05)
```

Networks are represented as NumPy `int16` arrays — 1 for an edge, 0 for no edge.

In [ ]:
SEED = 42
rng  = np.random.default_rng(SEED)

# Bipartite incidence matrix: rows = samples, cols = genes (or any two node classes).
# Each entry is 1 if the row-node is connected to the col-node, 0 otherwise.
bp   = (rng.random((100, 40)) < 0.20).astype(np.int16)
e_bp = int(bp.sum())  # total number of edges

print(f"Bipartite:  shape={bp.shape}, edges={e_bp}, density={e_bp/bp.size:.1%}")

In [ ]:
# Undirected adjacency matrix: symmetric, 0 on diagonal.
# We build it by sampling the upper triangle then mirroring.
_u  = (rng.random((200, 200)) < 0.05).astype(np.int16)
_u  = np.triu(_u, k=1)          # keep only upper triangle (no self-loops)
und = (_u + _u.T).clip(0, 1).astype(np.int16)  # mirror to make symmetric
e_und = int(und.sum()) // 2      # each edge counted twice in adjacency matrix

n_possible = und.shape[0] * (und.shape[0] - 1) // 2
print(f"Undirected: nodes={und.shape[0]}, edges={e_und}, density={e_und/n_possible:.1%}")

## 3. Bipartite rewiring

`rewire_bipartite` runs the Switching Algorithm for `max_iter` steps
(defaulting to the analytically derived bound N) and returns a new matrix
with identical row sums and column sums.

```r
# R equivalent
m2 <- birewire.rewire.bipartite(BRCA_binary_matrix, verbose=FALSE)
```

In [ ]:
# Rewire with the analytical bound N (default: max_iter='n').
# The seed makes the result reproducible; omit for random output.
bp_rw = pbr.rewire_bipartite(bp, verbose=False, seed=SEED)

# Degree preservation is guaranteed by the algorithm — these should always be True.
print("Row sums preserved:", np.array_equal(bp.sum(axis=1), bp_rw.sum(axis=1)))
print("Col sums preserved:", np.array_equal(bp.sum(axis=0), bp_rw.sum(axis=0)))

# Jaccard similarity measures edge overlap between original and rewired.
# At the stationary distribution, E[Jaccard] ≈ d / (2 - d) where d is edge density.
d = e_bp / bp.size
print(f"\nJaccard(original, rewired) = {jaccard(bp, bp_rw):.4f}")
print(f"Expected stationary Jaccard ≈ {d/(2-d):.4f}  (d / (2 - d), d={d:.3f})")

## 4. Analytical bound N

The bound N is the minimum number of switching steps needed for the chain to
be within accuracy δ of the stationary distribution:

$$N = \left\lceil \frac{e(1-d)}{2} \ln\frac{1-d}{\delta} \right\rceil$$

where $e$ is the number of edges and $d = e / (n_r \cdot n_c)$ is the density.
This is much tighter than the empirical bound from Milo et al. (2003).

In [ ]:
# bound_bipartite(n_edges, n_possible_edges, accuracy, exact)
# exact=False uses the asymptotic formula above; exact=True counts exact successful steps.
N_bp = bound_bipartite(e_bp, bp.size, accuracy=0.00005, exact=False)
print(f"Edges (e)       = {e_bp}")
print(f"Density (d)     = {e_bp/bp.size:.4f}")
print(f"Accuracy (δ)    = 0.00005")
print(f"Analytical N    = {N_bp}")
print(f"\nrewire_bipartite runs {N_bp} switching steps by default.")

## 5. Convergence analysis — bipartite

`analysis_bipartite` runs the SA independently `n_networks` times, recording
the Jaccard similarity to the original every `step` switching steps.
This lets you verify that N is conservative enough and that the chain has converged.

The plot reproduces the output of `birewire.analysis.bipartite` in the R vignette:
a linear-scale panel (top) and a log-log panel (bottom).

```r
res <- birewire.analysis.bipartite(get.incidence(ggg), max.iter=10*N, n.networks=10)
```

In [ ]:
# Run 10 independent chains, each for 10×N steps, sampling Jaccard every 10 steps.
# step=10 gives a dense curve; increase for speed on larger networks.
result_bp = pbr.analysis_bipartite(
    bp,
    n_networks = 10,       # independent rewiring runs for confidence interval
    max_iter   = 10 * N_bp,
    step       = 10,       # record Jaccard every 10 switching steps
    accuracy   = 0.00005,
    verbose    = False,
    seed       = SEED,
)

# result_bp.scores has shape (n_networks, n_steps)
# result_bp.N is the analytical bound (same as N_bp)
# result_bp.step is the sampling interval used
print(f"Scores shape: {result_bp.scores.shape}  (n_networks × n_steps)")
print(f"Bound N = {result_bp.N}")

In [ ]:
# Replicates the R vignette plot: two panels sharing the same data.
# x-axis starts at step (not 0) to match R's seq(1, length.out=n_steps) convention.
xs   = np.arange(1, result_bp.scores.shape[1] + 1) * result_bp.step
mean = result_bp.scores.mean(axis=0)
n_net = result_bp.scores.shape[0]
tval = t_dist.ppf(0.975, df=n_net - 1)   # t-quantile for 95% CI
se   = result_bp.scores.std(axis=0, ddof=1) / np.sqrt(n_net)
sup  = mean + tval * se
inf_ = mean - tval * se

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 8))

for ax, logscale in [(ax1, False), (ax2, True)]:
    # Gray ribbon = 95% confidence interval across the n_networks runs
    ax.fill_between(xs, inf_, sup, color="#CCCCCC", label="C.I.")
    # Blue line = mean Jaccard across runs
    ax.plot(xs, mean, color="blue", linewidth=2, label="Mean JI")
    # Red vertical line = analytical bound N
    ax.axvline(result_bp.N, color="red", linewidth=1, label="Bound")
    ax.set_xlabel("Switching steps", fontsize=10)
    ax.set_ylabel("Jaccard Index", fontsize=10)
    if logscale:
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_title("Jaccard index (JI) over time (log-log scale)",
                     fontsize=10, fontweight="bold")
        ax.legend(loc="lower left", fontsize=8)
    else:
        ax.set_title("Jaccard index (JI) over time",
                     fontsize=10, fontweight="bold")
        ax.legend(loc="upper right", fontsize=8)

plt.tight_layout()
plt.show()
# Expected: Jaccard starts at 1.0, drops sharply, then plateaus at ~d/(2-d).
# The red bound line should sit just before the plateau begins.

## 6. Undirected rewiring

`rewire_undirected` preserves the degree of every node (row sums of the
adjacency matrix). The output is always symmetric with a zero diagonal.

```r
# R equivalent
m2.und <- birewire.rewire.undirected(m.und, verbose=FALSE)
```

In [ ]:
und_rw = pbr.rewire_undirected(und, verbose=False, seed=SEED)

# For undirected graphs the degree sequence is the row-sum vector.
print("Degrees preserved:", np.array_equal(und.sum(axis=0), und_rw.sum(axis=0)))
print("Symmetric output: ", np.array_equal(und_rw, und_rw.T))
print("Zero diagonal:    ", und_rw.diagonal().sum() == 0)
print(f"Jaccard(original, rewired) = {jaccard(und, und_rw):.4f}")

## 7. Convergence analysis — undirected

Identical workflow to the bipartite case; the bound formula is the same
with $n_r \cdot n_c$ replaced by the number of possible edges $n(n-1)/2$.

In [ ]:
n_possible_und = und.shape[0] * (und.shape[0] - 1) // 2
N_und = bound_bipartite(e_und, n_possible_und, accuracy=0.00005, exact=False)
print(f"Undirected N = {N_und}")

result_und = pbr.analysis_undirected(
    und,
    n_networks = 5,
    max_iter   = 10 * N_und,
    step       = 10,
    verbose    = False,
    seed       = SEED,
)
print(f"Scores shape: {result_und.scores.shape}")

In [ ]:
xs_u  = np.arange(1, result_und.scores.shape[1] + 1) * result_und.step
mean_u = result_und.scores.mean(axis=0)
n_u    = result_und.scores.shape[0]
tval_u = t_dist.ppf(0.975, df=n_u - 1)
se_u   = result_und.scores.std(axis=0, ddof=1) / np.sqrt(n_u)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 8))

for ax, logscale in [(ax1, False), (ax2, True)]:
    ax.fill_between(xs_u, mean_u - tval_u*se_u, mean_u + tval_u*se_u,
                    color="#CCCCCC", label="C.I.")
    ax.plot(xs_u, mean_u, color="blue", linewidth=2, label="Mean JI")
    ax.axvline(result_und.N, color="red", linewidth=1, label="Bound")
    ax.set_xlabel("Switching steps", fontsize=10)
    ax.set_ylabel("Jaccard Index", fontsize=10)
    if logscale:
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_title("Jaccard index (JI) over time (log-log scale)",
                     fontsize=10, fontweight="bold")
        ax.legend(loc="lower left", fontsize=8)
    else:
        ax.set_title("Jaccard index (JI) over time",
                     fontsize=10, fontweight="bold")
        ax.legend(loc="upper right", fontsize=8)

plt.tight_layout()
plt.show()

## 8. Input format support

All rewiring functions accept multiple input formats and **return the same type
as the input** — no conversion needed in your pipeline.

| Input | Output |
|-------|--------|
| `numpy.ndarray` | `numpy.ndarray` |
| `scipy.sparse` (any format) | `scipy.sparse.csr_matrix` |
| `igraph.Graph` | `igraph.Graph` |
| `networkx.Graph` | `networkx.Graph` |

In [ ]:
import scipy.sparse as sp

# Convert the bipartite matrix to a sparse format — useful for large, low-density networks.
bp_csr = sp.csr_matrix(bp)
print(f"Input:  {type(bp_csr).__name__}, nnz={bp_csr.nnz}")

rw_csr = pbr.rewire_bipartite(bp_csr, verbose=False, seed=SEED)
print(f"Output: {type(rw_csr).__name__}, nnz={rw_csr.nnz}")

# Degree preservation checked on the sparse matrix directly
rows_ok = np.array_equal(
    np.asarray(bp_csr.sum(axis=1)).ravel(),
    np.asarray(rw_csr.sum(axis=1)).ravel()
)
print(f"Row sums preserved: {rows_ok}")

# CSC format is also accepted
bp_csc = bp_csr.tocsc()
rw_csc = pbr.rewire_bipartite(bp_csc, verbose=False, seed=SEED)
print(f"\nCSC input  → {type(rw_csc).__name__} output")

In [ ]:
try:
    import igraph as ig

    # Build a bipartite igraph graph with the same structure as bp.
    # types: False = row nodes (100), True = col nodes (40).
    types = [False] * 100 + [True] * 40
    edges = [(i, 100 + j)
             for i in range(100)
             for j in range(40)
             if bp[i, j] == 1]
    g_bp = ig.Graph.Bipartite(types, edges)
    print(f"igraph input:  {g_bp.vcount()} vertices, {g_bp.ecount()} edges")

    # rewire_bipartite detects the igraph object and returns an igraph.Graph
    g_rw = pbr.rewire_bipartite(g_bp, verbose=False, seed=SEED)
    print(f"igraph output: {g_rw.vcount()} vertices, {g_rw.ecount()} edges")
    print(f"Type preserved: {type(g_rw).__name__}")

    # Degree sequence is preserved
    deg_ok = g_bp.degree() == g_rw.degree()
    print(f"Degrees preserved: {all(deg_ok)}")

except ImportError:
    print("igraph not installed. Install with: pip install 'pybirewirex[graph]'")

In [ ]:
try:
    import networkx as nx

    # Build a bipartite NetworkX graph from the same incidence matrix.
    g_nx = nx.Graph()
    g_nx.add_nodes_from(range(100), bipartite=0)          # row nodes
    g_nx.add_nodes_from(range(100, 140), bipartite=1)     # col nodes
    g_nx.add_edges_from(
        (i, 100 + j)
        for i in range(100)
        for j in range(40)
        if bp[i, j] == 1
    )
    print(f"NetworkX input:  {g_nx.number_of_nodes()} nodes, {g_nx.number_of_edges()} edges")

    # rewire_bipartite detects networkx.Graph and returns networkx.Graph
    g_nx_rw = pbr.rewire_bipartite(g_nx, verbose=False, seed=SEED)
    print(f"NetworkX output: {g_nx_rw.number_of_nodes()} nodes, {g_nx_rw.number_of_edges()} edges")
    print(f"Type preserved: {type(g_nx_rw).__name__}")

    # Edge count preserved (degree-preserving rewiring never adds or removes edges)
    print(f"Edge count preserved: {g_nx.number_of_edges() == g_nx_rw.number_of_edges()}")

except ImportError:
    print("networkx not installed. Install with: pip install 'pybirewirex[graph]'")

## 9. Null-model sampler

To generate a collection of K null-model networks, rewire sequentially:
each sample is N switching steps beyond the previous one.
This matches R's `birewire.sampler.bipartite`.

```r
birewire.sampler.bipartite(ggg, K=10000, path="TESTBIREWIREBIPARTITE")
```

All K samples are guaranteed to be degree-preserving and approximately
independent (each separated by N steps in the chain).

In [ ]:
K       = 10
samples = []
current = bp.copy()

for k in range(K):
    # Each call advances the chain by N steps from the previous state.
    # max_iter='n' (default) automatically uses the analytical bound.
    current = pbr.rewire_bipartite(current, verbose=False, seed=SEED + k + 1)
    samples.append(current.copy())

# Every sample must have the same row and column sums as the original
all_ok = all(
    np.array_equal(s.sum(axis=1), bp.sum(axis=1)) and
    np.array_equal(s.sum(axis=0), bp.sum(axis=0))
    for s in samples
)
print(f"{K} samples generated, all degree-preserving: {all_ok}")

# Pairwise Jaccard between samples quantifies how different they are from each other.
# Values close to d/(2-d) ≈ 0.11 indicate the samples are well-spread in network space.
pw = [jaccard(samples[i], samples[j]) for i in range(K) for j in range(i+1, K)]
print(f"Pairwise Jaccard: mean={np.mean(pw):.3f}  min={np.min(pw):.3f}  max={np.max(pw):.3f}")
print(f"Expected stationary Jaccard ≈ {d/(2-d):.3f}")

## 10. Markov chain monitoring

This visualises how the Markov chain explores network space as the number of
switching steps increases.

For each step count k, we generate `n_networks` networks by walking the chain
k steps at a time. Pairwise Jaccard distances are projected to 2D via t-SNE.

- **Small k:** chain moves slowly → networks cluster tightly (little diversity)
- **k ≈ N:** chain has mixed → networks spread uniformly (maximum diversity)
- **k >> N:** same uniform distribution, just slower to compute

This reproduces the output of `birewire.visual.monitoring.bipartite` in the R vignette.

```r
tsne = birewire.visual.monitoring.bipartite(
           ggg, display=T, n.networks=75,
           sequence=c(1, 10, 200, 1000, "n", 50000), ncol=3, perplexity=10)
```

In [ ]:
def collect_chain(start, interval, n_samples, base_seed):
    """Walk the Markov chain: each network is `interval` steps beyond the previous."""
    nets    = [start.copy()]
    current = start.copy()
    for i in range(n_samples - 1):
        current = pbr.rewire_bipartite(
            current, max_iter=interval, verbose=False, seed=base_seed + i)
        nets.append(current.copy())
    return nets


def pairwise_dist(nets):
    """Compute pairwise Jaccard distance matrix (1 - Jaccard similarity)."""
    n = len(nets)
    D = np.zeros((n, n))
    for i in range(n):
        for j in range(i + 1, n):
            D[i, j] = D[j, i] = 1.0 - jaccard(nets[i], nets[j])
    return D


def embed_tsne(D, perplexity, seed):
    """Replicate R's Rtsne pipeline: normalize → PCA(50) → t-SNE(euclidean)."""
    D_n   = D / (np.abs(D).max() + 1e-12)           # global max-abs normalization
    n_pca = min(50, D_n.shape[1] - 1)
    D_pca = PCA(n_components=n_pca, random_state=seed).fit_transform(D_n)
    return TSNE(
        n_components=2, metric="euclidean",
        perplexity=perplexity, random_state=seed,
        init="random", max_iter=1000,
    ).fit_transform(D_pca)

In [ ]:
# n_networks=30 for notebook speed; scripts/demo.py uses 75 (matches R vignette exactly).
N_MON    = 30
PERP     = 6       # perplexity must be < n_networks; R uses 10 with n=75
SEQUENCE = [1, 10, 200, 1000, N_bp, 50000]
LABELS   = ["1", "10", "200", "1000", "N", "50000"]

embeddings = []
for k, (iv, lbl) in enumerate(zip(SEQUENCE, LABELS)):
    print(f"k={lbl:<7} collecting {N_MON} networks … ", end="", flush=True)
    nets = collect_chain(bp, iv, N_MON, base_seed=SEED + k * 10000)
    D    = pairwise_dist(nets)
    embeddings.append(embed_tsne(D, perplexity=PERP, seed=SEED))
    print("done")

In [ ]:
cmap = plt.cm.coolwarm
norm = Normalize(vmin=0, vmax=N_MON - 1)

fig, axes = plt.subplots(2, 3, figsize=(11, 7))

for ax, emb, lbl in zip(axes.ravel(), embeddings, LABELS):
    colors = cmap(norm(np.arange(N_MON)))

    # All networks except the first (coloured blue→red by sampling order)
    ax.scatter(emb[1:, 0], emb[1:, 1], c=colors[1:], s=12, linewidths=0)

    # First network (the original) marked with a black border and "start" label
    ax.scatter(emb[0, 0], emb[0, 1], color=colors[0], s=60,
               edgecolors="black", linewidths=0.8)
    ax.text(emb[0, 0], emb[0, 1], " start", fontsize=8)

    ax.set_title(f"k= {lbl}", fontsize=10, fontweight="bold", pad=3)
    ax.set_xlabel("A.U.", fontsize=8)
    ax.set_ylabel("A.U.", fontsize=8)
    ax.tick_params(labelsize=7)
    for sp in ax.spines.values():
        sp.set_linewidth(0.5)

fig.subplots_adjust(left=0.07, right=0.97, top=0.95, bottom=0.08,
                    hspace=0.45, wspace=0.35)
plt.show()
# Expected: k=1 → tight cluster (few steps, chain barely moved).
#           k=N → scattered cloud (chain mixed, samples uniformly distributed).
#           k=50000 → same scattered cloud as k=N (already at stationarity).

---

## Citation

If you use PyBiRewireX in published research, please cite the original papers:

> Gobbi, A. et al. (2014). *Fast randomization of large genomic datasets while
> preserving alteration counts.* Bioinformatics, 30(17), i617–i623.
> https://doi.org/10.1093/bioinformatics/btu474

> Iorio, F. et al. (2016). *Efficient randomization of biological networks while
> preserving functional characterization of individual nodes.*
> BMC Bioinformatics, 17, 542.
> https://doi.org/10.1186/s12859-016-1402-1